In [28]:
from langgraph.graph import StateGraph, START, END
from langchain_ollama import ChatOllama

from pydantic import BaseModel, Field
from typing import TypedDict, Literal


In [29]:
llm = ChatOllama(
    model= "llama3.2:1b"
)

In [ ]:
class sentimentSchema(BaseModel):
    sentiment: Literal["positive", "negative"] = Field(description="Sentiment of the review")

class DiagnosisSchema(BaseModel):
    issue_type: Literal["UX", "Performance", "Bug", "Support", "Other"] = Field(description='The category of issue mentioned in the review')
    tone: Literal["angry", "frustrated", "disappointed", "calm"] = Field(description='The emotional tone expressed by the user')
    urgency: Literal["low", "medium", "high"] = Field(description='How urgent or critical the issue appears to be')

In [31]:
structured_model = llm.with_structured_output(sentimentSchema)
structured_model2 = llm.with_structured_output(DiagnosisSchema)

structured_model.invoke("What is the sentiment of the following: \n The software is too good")

sentimentSchema(sentiment='positive')

In [32]:
class ReviewState(TypedDict):
    review: str
    sentiment: Literal["positive", "negative"]
    diagnosis: dict
    response: str

In [33]:
def find_sentiment(state: ReviewState):
    prompt = f'For the following review find out the sentiment \n {state["review"]}'
    sentiment = structured_model.invoke(prompt).sentiment

    return {"sentiment": sentiment}

def checkSentiment(state: ReviewState) -> Literal["positive_response", "run_diagnosis"]:
    if state["sentiment"] == "positive":
        return "positive_response"
    else:
        return "run_diagnosis"

def positive_response(state: ReviewState):

    prompt = f"""Write a warm thank-you message in response to this review:
    \n\n\"{state['review']}\"\n
Also, kindly ask the user to leave feedback on our website."""
    
    response = llm.invoke(prompt).content

    return {'response': response}

def run_diagnosis(state: ReviewState):

    prompt = f"""Diagnose this negative review:\n\n{state['review']}\n"
    "Return issue_type, tone, and urgency.
"""
    response = structured_model2.invoke(prompt)

    return {'diagnosis': response.model_dump()}

def negative_response(state: ReviewState):

    diagnosis = state['diagnosis']

    prompt = f"""You are a support assistant.
The user had a '{diagnosis['issue_type']}' issue, sounded '{diagnosis['tone']}', and marked urgency as '{diagnosis['urgency']}'.
Write an empathetic, helpful resolution message.
"""
    response = llm.invoke(prompt).content

    return {'response': response}

    

In [34]:
graph = StateGraph(ReviewState)

graph.add_node("find_sentiment", find_sentiment)
graph.add_node("positive_response", positive_response)
graph.add_node("run_diagnosis", run_diagnosis)
graph.add_node("negative_response", negative_response)

graph.add_edge(START, "find_sentiment")
graph.add_conditional_edges("find_sentiment", checkSentiment)

graph.add_edge("positive_response", END)
graph.add_edge("run_diagnosis", "negative_response")
graph.add_edge("negative_response", END)

workflow = graph.compile()

workflow.invoke({"review": "The product was more than okay"})



{'review': 'The product was more than okay',
 'sentiment': 'negative',
 'diagnosis': {'issue_type': 'Performance',
  'tone': 'disappointed',
  'urgency': 'low'},
 'response': 'Here\'s a possible response:\n\n"I\'m so sorry to hear that you\'re experiencing a performance issue and feeling disappointed. I can imagine how frustrating and concerning that must be for you.\n\nI want to assure you that we\'re here to help and support you. I\'ve logged a ticket for you to address the issue, and I\'ll make sure to provide you with any necessary assistance and updates on the status of the ticket.\n\nIn the meantime, I\'d like to offer some general guidance to help you troubleshoot the issue. Can you please provide me with more details about the problem you\'re experiencing, such as any error messages or steps you\'ve taken so far? This will help me to better understand the issue and provide you with more tailored support.\n\nRegarding the urgency level, I understand that you initially marked it 